# Method 2 completion 2: round1
Attach the strict completion bundle. See docs/method2_completion.md for the required previous outputs. GPU T4, fresh session. Test results must not select hyperparameters.


In [1]:
from pathlib import Path
import os, shutil, subprocess, sys, json
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
bundles = list(Path("/kaggle/input").rglob("completion_manifest.json"))
if len(bundles) != 1:
    raise RuntimeError(f"Attach exactly one completion bundle; found {len(bundles)}")
bundle = bundles[0].parent
work = Path("/kaggle/working")
for name in ["src", "scripts", "configs", "data", "docs", "notebooks"]:
    shutil.copytree(bundle / name, work / name, dirs_exist_ok=True)
shutil.copy2(bundle / "same_domain_feasibility.json", work / "same_domain_feasibility.json")
shutil.copy2(bundle / "completion_manifest.json", work / "completion_manifest.json")
os.chdir(work)
caches = list(Path("/kaggle/input").rglob("models--BAAI--bge-m3"))
if caches:
    cache_hub = caches[0].parent
    os.environ["HF_HUB_CACHE"] = str(cache_hub)
    os.environ["HF_HOME"] = str(cache_hub.parent)
pins = json.loads(Path("configs/method2/pinned_versions.json").read_text())["pinned"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *[f"{k}=={v}" for k,v in pins.items()], "jsonschema", "pyyaml", "matplotlib", "rank_bm25", "datasets", "accelerate"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
lora_smoke = """import torch
from transformers import XLMRobertaConfig, XLMRobertaModel
from peft import LoraConfig
config = XLMRobertaConfig(vocab_size=32, hidden_size=8, num_hidden_layers=1, num_attention_heads=2, intermediate_size=16, max_position_embeddings=32)
model = XLMRobertaModel(config)
model.add_adapter(LoraConfig(r=2, lora_alpha=4, target_modules=["query", "value"]))
model(torch.tensor([[0, 5, 2]])).last_hidden_state.square().mean().backward()
assert any(p.grad is not None for n, p in model.named_parameters() if "lora_" in n)
print("LoRA environment smoke passed")
"""
subprocess.run([sys.executable, "-c", lora_smoke], check=True)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 101.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.6/739.6 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 33.4 MB/s eta 0:00:00
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
LoRA environment smoke passed


CompletedProcess(args=['/usr/bin/python3', '-c', 'import torch\nfrom transformers import XLMRobertaConfig, XLMRobertaModel\nfrom peft import LoraConfig\nconfig = XLMRobertaConfig(vocab_size=32, hidden_size=8, num_hidden_layers=1, num_attention_heads=2, intermediate_size=16, max_position_embeddings=32)\nmodel = XLMRobertaModel(config)\nmodel.add_adapter(LoraConfig(r=2, lora_alpha=4, target_modules=["query", "value"]))\nmodel(torch.tensor([[0, 5, 2]])).last_hidden_state.square().mean().backward()\nassert any(p.grad is not None for n, p in model.named_parameters() if "lora_" in n)\nprint("LoRA environment smoke passed")\n'], returncode=0)

In [2]:
prior_stage = None
if prior_stage:
    markers = list(Path("/kaggle/input").rglob(f"{prior_stage}_completed.json"))
    if len(markers) != 1:
        raise RuntimeError(f"Attach exactly one strict {prior_stage} output, including results/; found {len(markers)}")
    previous = markers[0].parents[3]
    for name in ["artifacts/method2", "results/method2/completion", "data/method2/index"]:
        if (previous / name).exists():
            shutil.copytree(previous / name, work / name, dirs_exist_ok=True, ignore=shutil.ignore_patterns("checkpoint-*"))
if False:
    validation_markers = list(Path("/kaggle/input").rglob("validation_completed.json"))
    if len(validation_markers) != 1:
        raise RuntimeError("Attach exactly one validation output for the selected CE checkpoint")
    validation_root = validation_markers[0].parents[3]
    shutil.copytree(validation_root / "results/method2/completion/validation", work / "results/method2/completion/validation", dirs_exist_ok=True)
    shutil.copy2(validation_markers[0], work / "results/method2/completion/validation_completed.json")
    shutil.copytree(validation_root / "artifacts/method2/crossencoder", work / "artifacts/method2/crossencoder", dirs_exist_ok=True, ignore=shutil.ignore_patterns("checkpoint-*"))
ce = Path("artifacts/method2/crossencoder/run01/final")
if False and not (ce / "crossencoder_heads.pt").exists():
    heads = [p for p in Path("/kaggle/input").rglob("crossencoder_heads.pt") if p.parent.name == "final"]
    if len(heads) != 1:
        raise RuntimeError(f"Attach one CE final checkpoint (model + heads + tokenizer), found {len(heads)}")
    shutil.copytree(heads[0].parent, ce, dirs_exist_ok=True)


In [3]:
subprocess.run([sys.executable, "scripts/method2/run_completion.py", "round1"], check=True)
print(Path("results/method2/completion/round1_completed.json").read_text(encoding="utf-8")[:2000])


[biencoder] CUDA_VISIBLE_DEVICES=0 — ép chạy một GPU
[biencoder] TensorFlow version 2.20.0 available.
[biencoder] JAX version 0.7.2 available.
/kaggle/working/src/models/biencoder/train.py:223: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers.losses import CachedMultipleNegativesRankingLoss
[biencoder] train pairs: 78435
[biencoder] No device provided, using cuda:0
[biencoder] HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
[biencoder] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
[biencoder] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/modules.json "HTTP/1.1 200 OK"
[biencoder] HTTP Req

{'loss': '3.86', 'grad_norm': '2.8', 'learning_rate': '1.054e-05', 'epoch': '0.1629'}
{'loss': '2.939', 'grad_norm': '1.892', 'learning_rate': '2e-05', 'epoch': '0.3257'}


 22%|██▏       | 200/921 [1:56:48<7:01:27, 35.07s/it][biencoder] Saving model checkpoint to artifacts/method2/biencoder/strict_round1/checkpoint-200
[biencoder] Saving model to artifacts/method2/biencoder/strict_round1/checkpoint-200
[biencoder] HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
[biencoder] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/config.json "HTTP/1.1 200 OK"
[biencoder] HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
[biencoder] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/config.json "HTTP/1.1 200 OK"

Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 18.60it/s]


{'loss': '2.18', 'grad_norm': '2.042', 'learning_rate': '1.978e-05', 'epoch': '0.4886'}
{'loss': '1.992', 'grad_norm': '2.107', 'learning_rate': '1.92e-05', 'epoch': '0.6515'}


 33%|███▎      | 300/921 [2:55:35<6:00:44, 34.85s/it][biencoder] Information Retrieval Evaluation of the model on the custom_val dataset in epoch 0.9771986970684039 after 300 steps:

Batches:   0%|          | 1/308 [00:00<00:47,  6.42it/s]

...

{'eval_custom_val_cosine_accuracy@1': '0.5535', 'eval_custom_val_cosine_accuracy@3': '0.7828', 'eval_custom_val_cosine_accuracy@5': '0.8513', 'eval_custom_val_cosine_accuracy@10': '0.9103', 'eval_custom_val_cosine_precision@1': '0.5535', 'eval_custom_val_cosine_precision@3': '0.2609', 'eval_custom_val_cosine_precision@5': '0.1703', 'eval_custom_val_cosine_precision@10': '0.09103', 'eval_custom_val_cosine_recall@1': '0.5535', 'eval_custom_val_cosine_recall@3': '0.7828', 'eval_custom_val_cosine_recall@5': '0.8513', 'eval_custom_val_cosine_recall@10': '0.9103', 'eval_custom_val_cosine_ndcg@10': '0.7367', 'eval_custom_val_cosine_mrr@10': '0.6804', 'eval_custom_val_cosine_map@100': '0.6844', 'eval_runtime': '47.62', 'eval_samples_per_second': '0', 'eval_steps_per_second': '0', 'epoch': '3'}


100%|██████████| 921/921 [9:01:57<00:00, 35.31s/it]
[biencoder] Saving model to artifacts/method2/biencoder/strict_round1/final
[biencoder] HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
[biencoder] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/config.json "HTTP/1.1 200 OK"
[biencoder] HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
[biencoder] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/config.json "HTTP/1.1 200 OK"
Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 20.70it/s]


{'train_runtime': '3.252e+04', 'train_samples_per_second': '7.236', 'train_steps_per_second': '0.028', 'train_loss': '1.945', 'epoch': '3'}


[biencoder] Information Retrieval Evaluation of the model on the custom_val dataset:
Corpus Chunks: 100%|██████████| 1/1 [00:14<00:00, 14.83s/it]
[biencoder] Queries: 9846
[biencoder] Corpus: 4464

[biencoder] Score-Function: cosine
[biencoder] Accuracy@1: 55.35%
[biencoder] Accuracy@3: 78.28%
[biencoder] Accuracy@5: 85.13%
[biencoder] Accuracy@10: 91.03%
[biencoder] Precision@1: 55.35%
[biencoder] Precision@3: 26.09%
[biencoder] Precision@5: 17.03%
[biencoder] Precision@10: 9.10%
[biencoder] Recall@1: 55.35%
[biencoder] Recall@3: 78.28%
[biencoder] Recall@5: 85.13%
[biencoder] Recall@10: 91.03%
[biencoder] MRR@10: 0.6804
[biencoder] NDCG@10: 0.7367
[biencoder] MAP@100: 0.6844


{
  "custom_val_cosine_accuracy@1": 0.5535242738167784,
  "custom_val_cosine_accuracy@3": 0.7827544180377818,
  "custom_val_cosine_accuracy@5": 0.8513101767215113,
  "custom_val_cosine_accuracy@10": 0.910318911232988,
  "custom_val_cosine_precision@1": 0.5535242738167784,
  "custom_val_cosine_precision@3": 0.2609181393459273,
  "custom_val_cosine_precision@5": 0.17026203534430226,
  "custom_val_cosine_precision@10": 0.0910318911232988,
  "custom_val_cosine_recall@1": 0.5535242738167784,
  "custom_val_cosine_recall@3": 0.7827544180377818,
  "custom_val_cosine_recall@5": 0.8513101767215113,
  "custom_val_cosine_recall@10": 0.910318911232988,
  "custom_val_cosine_ndcg@10": 0.7366708447716995,
  "custom_val_cosine_mrr@10": 0.680414857052571,
  "custom_val_cosine_map@100": 0.6843959949173296
}
{
  "metric": null,
  "resolved_metric": "eval_custom_val_cosine_ndcg@10",
  "best_value": 0.7366880349523601,
  "best_step": 900,
  "n_evaluations": 4,
  "available_metrics": [
    "eval_custom_val_c

In [4]:
archive = work / "method2_completion_round1_reports.tar.gz"
items = [name for name in ["results/method2/completion", "data/method2/biencoder/train_mined.jsonl"] if Path(name).exists()]
subprocess.run(["tar", "czf", str(archive), *items], check=True)
print(archive, archive.stat().st_size)
print("Save the whole notebook output for the next stage; this small archive does not contain model weights or the index.")


/kaggle/working/method2_completion_round1_reports.tar.gz 946
Save the whole notebook output for the next stage; this small archive does not contain model weights or the index.
